# YOLO11 객체인식 + TensorRT Quantization 실습

**Author**: Hyun-seo In (Hanyang Univ. BME/AE) — inhsroy@hanyang.ac.kr  
**Date**: 2026.05 | Jetson Orin Nano 8GB / JetPack 6.x

## 목차
1. 환경 확인
2. 라이브러리 설치 (Ultralytics)
3. CSI 카메라로 프레임 캡처
4. YOLO11 PyTorch 추론 (베이스라인)
5. ONNX 변환
6. TensorRT 엔진 빌드
7. TensorRT 엔진으로 추론
8. 성능 비교 (FP32 / FP16 / INT8)

## 1. 환경 확인

JetPack 설치 시 PyTorch / OpenCV / TensorRT 모두 자동 포함됨. 버전 확인부터.

In [ ]:
!python3 --version
!python3 -c "import torch; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"
!python3 -c "import cv2; print('opencv:', cv2.__version__)"
!python3 -c "import tensorrt; print('tensorrt:', tensorrt.__version__)"

## 2. 라이브러리 설치 (Ultralytics)

> ⚠️ Jetson 주의사항
> - **OpenCV**: `pip install opencv-python` 금지 (apt 버전이 GStreamer/CUDA 지원)
> - **NumPy**: 2.x 호환성 깨짐 → 1.x 유지 필요
> - ultralytics 설치 시 numpy/opencv를 덮어쓰므로 설치 후 다운그레이드

In [ ]:
%pip install ultralytics

In [ ]:
# ultralytics가 설치한 pip opencv 제거 (apt 버전 사용)
%pip uninstall opencv-python -y

# numpy 1.x로 다운그레이드
%pip install "numpy<2.0"

# 'opencv-python not installed' 경고는 무시 가능 (apt cv2 사용 중)

In [ ]:
# 최종 확인
!python3 -c "import numpy; print('numpy:', numpy.__version__)"
!python3 -c "import cv2; print('opencv:', cv2.__version__)"
!python3 -c "from ultralytics import YOLO; print('ultralytics ok')"

## 3. CSI 카메라로 프레임 캡처

> ⚠️ Remote SSH 환경에서는 `cv2.imshow` 동작 안 함  
> GStreamer로 단일 프레임 캡처 → 파일 저장 → 추론 입력으로 사용

### 카메라 인식 확인

In [ ]:
!ls /dev/video*
# /dev/video0 출력되면 정상

### 단일 프레임 캡처 → test.jpg 저장

In [ ]:
!gst-launch-1.0 nvarguscamerasrc num-buffers=1 ! \
  'video/x-raw(memory:NVMM), width=1920, height=1080, format=NV12, framerate=30/1' ! \
  nvjpegenc ! filesink location=test.jpg
!ls -lh test.jpg

## 4. YOLO11 PyTorch 추론 (베이스라인)

> 💡 모델 크기: n < s < m < l < x  
> Jetson Orin Nano 8GB 기준 m까지 실시간 추론 가능. 본 실습은 nano 사용.

In [ ]:
from ultralytics import YOLO
import numpy as np

# 모델 자동 다운로드
model = YOLO("yolo11n.pt")
print("model loaded |", "classes:", len(model.names))

# 더미 추론 테스트
dummy = np.zeros((640, 640, 3), dtype=np.uint8)
results = model(dummy, verbose=False)
print("inference ok")

### 캡처한 이미지로 추론 → 결과 저장

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
img = cv2.imread("test.jpg")
results = model(img, verbose=False)
annotated = results[0].plot()

# VS Code 탐색기에서 result_fp32.jpg 더블클릭 → 결과 확인
cv2.imwrite("result_fp32.jpg", annotated)
print(f"detections: {len(results[0].boxes)}")
print("saved → result_fp32.jpg")

### Latency 측정 (더미 입력 100회)

In [ ]:
import torch, time
import numpy as np
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
dummy = np.zeros((640, 640, 3), dtype=np.uint8)

# warmup
for _ in range(10):
    model(dummy, verbose=False)

# 측정
times = []
for _ in range(100):
    start = time.time()
    model(dummy, verbose=False)
    torch.cuda.synchronize()
    times.append((time.time() - start) * 1000)

avg_ms = sum(times) / len(times)
print(f"[FP32] Latency: {avg_ms:.2f} ms | FPS: {1000/avg_ms:.1f}")

## 5. ONNX 변환

PyTorch `.pt` → ONNX `.onnx` (TensorRT가 읽을 수 있는 표준 포맷)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.export(format="onnx", imgsz=640, opset=17, simplify=True)

In [ ]:
!ls -lh yolo11n.onnx

## 6. TensorRT 엔진 빌드

> ⚠️ 처음 빌드는 **5~10분** 소요  
> 빌드된 엔진은 해당 Jetson 하드웨어 전용 (다른 기기 이식 불가)

In [ ]:
!mkdir -p engines logs

### FP16 엔진 빌드

In [ ]:
!/usr/src/tensorrt/bin/trtexec \
  --onnx=yolo11n.onnx \
  --saveEngine=engines/yolo11n_fp16.engine \
  --fp16 \
  --separateProfileRun \
  --profilingVerbosity=detailed \
  2>&1 | tee logs/yolo_fp16.txt

### INT8 엔진 빌드

In [ ]:
!/usr/src/tensorrt/bin/trtexec \
  --onnx=yolo11n.onnx \
  --saveEngine=engines/yolo11n_int8.engine \
  --int8 --fp16 \
  --separateProfileRun \
  --profilingVerbosity=detailed \
  2>&1 | tee logs/yolo_int8.txt

In [ ]:
!ls -lh engines/

## 7. TensorRT 엔진으로 추론

ultralytics가 `.engine` 파일 직접 로드 지원.

### FP16 엔진 추론

In [ ]:
import cv2
from ultralytics import YOLO

model_fp16 = YOLO("engines/yolo11n_fp16.engine")
img = cv2.imread("test.jpg")
results = model_fp16(img, verbose=False)
annotated = results[0].plot()
cv2.imwrite("result_fp16.jpg", annotated)
print(f"FP16 detections: {len(results[0].boxes)} → result_fp16.jpg")

### INT8 엔진 추론

In [ ]:
import cv2
from ultralytics import YOLO

model_int8 = YOLO("engines/yolo11n_int8.engine")
img = cv2.imread("test.jpg")
results = model_int8(img, verbose=False)
annotated = results[0].plot()
cv2.imwrite("result_int8.jpg", annotated)
print(f"INT8 detections: {len(results[0].boxes)} → result_int8.jpg")

## 8. 성능 비교 (FP32 / FP16 / INT8)

In [ ]:
import cv2, time
from ultralytics import YOLO

models = {
    "FP32": YOLO("yolo11n.pt"),
    "FP16": YOLO("engines/yolo11n_fp16.engine"),
    "INT8": YOLO("engines/yolo11n_int8.engine"),
}

img = cv2.imread("test.jpg")

print(f"{'Backend':<8} {'Latency (ms)':<15} {'FPS':<10}")
print("-" * 35)
for name, model in models.items():
    # warmup
    for _ in range(5):
        model(img, verbose=False)
    # 측정
    times = []
    for _ in range(50):
        start = time.time()
        model(img, verbose=False)
        times.append((time.time() - start) * 1000)
    avg_ms = sum(times) / len(times)
    print(f"{name:<8} {avg_ms:<15.2f} {1000/avg_ms:<10.1f}")

### 정확도 차이 시각 비교 (이미지 저장)

FP32 / FP16 / INT8 결과를 좌우로 이어붙여 한 장 이미지로 저장

In [ ]:
import cv2
from ultralytics import YOLO

img = cv2.imread("test.jpg")
fp32 = YOLO("yolo11n.pt")(img, verbose=False)[0].plot()
fp16 = YOLO("engines/yolo11n_fp16.engine")(img, verbose=False)[0].plot()
int8 = YOLO("engines/yolo11n_int8.engine")(img, verbose=False)[0].plot()

# 좌우로 이어붙이기
compare = cv2.hconcat([fp32, fp16, int8])
cv2.imwrite("compare_fp32_fp16_int8.jpg", compare)
print("saved → compare_fp32_fp16_int8.jpg")

### 예상 성능 (실측값으로 교체 예정)

| Backend | Latency (ms) | FPS | 비고 |
|---------|-------------|-----|------|
| PyTorch FP32 | ~35 | ~28 | 베이스라인 |
| TensorRT FP16 | ~12 | ~83 | 정확도 유지 |
| TensorRT INT8 | ~8 | ~125 | 정확도 소폭 감소 |